# BERT + LoRA: fixed hyperparameter search

In [ ]:
%cd /content
!rm -rf LORA-course-project
!git clone https://github.com/NikitaBaranenkov/LORA-course-project.git
%cd LORA-course-project
!ls
!ls src

/content
Cloning into 'LORA-course-project'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 90 (delta 21), reused 68 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 685.60 KiB | 7.45 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/LORA-course-project
artifacts  data    notebooks  reports		src
configs    legacy  README.md  requirements.txt
constants.py  __init__.py  README.md


In [ ]:
%pip uninstall -y torchao
%pip install -q --upgrade peft transformers accelerate datasets evaluate scikit-learn

In [ ]:
import gc
import inspect
import json
import random
import time
from itertools import product

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from src.constants import (
    BASE_MODEL_BERT,
    DATASET_NAME,
    FINAL_TEST_EVALUATION_ONLY,
    MAX_LENGTH,
    PROJECT_ROOT,
    REPORTS_TABLES_DIR,
    SEED,
    SELECTION_METRIC,
    TEST_CSV_PATH,
    TRAIN_CSV_PATH,
    VALIDATION_CSV_PATH,
    id2label,
    label2id,
)

RUN_NAME = "bert_lora_hparam_search"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / RUN_NAME
CHECKPOINTS_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_ADAPTER_DIR = ARTIFACTS_DIR / "best_adapter"
RUN_TABLES_DIR = REPORTS_TABLES_DIR / "bert_lora"

SWEEP_RESULTS_PATH = RUN_TABLES_DIR / "bert_lora_hparam_search_validation_results.csv"
FINAL_TEST_RESULTS_PATH = RUN_TABLES_DIR / "bert_lora_best_hparam_test_results.csv"
EXPERIMENT_CONFIG_PATH = RUN_TABLES_DIR / "bert_lora_hparam_search_config.json"


## Импорты и настройка окружения

## Загрузка фиксированных разбиений данных

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH).reset_index(drop=True)
validation_df = pd.read_csv(VALIDATION_CSV_PATH).reset_index(drop=True)
test_df = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)

for split_df in (train_df, validation_df, test_df):
    split_df["label"] = split_df["label"].astype(int)

raw_dataframes = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

raw_datasets = DatasetDict({
    split_name: Dataset.from_pandas(split_df, preserve_index=False)
    for split_name, split_df in raw_dataframes.items()
})
raw_datasets = raw_datasets.rename_column("label", "labels")

raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 8105
    })
    validation: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 1431
    })
    test: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 2388
    })
})

## Токенизация текста

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_BERT)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label_name"],
)
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

sample = tokenized_datasets["train"][0]
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/8105 [00:00<?, ? examples/s]

Map:   0%|          | 0/1431 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
label: 1


## Метрики качества

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


def softmax_numpy(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    probabilities = np.exp(shifted_logits)
    return probabilities / np.sum(probabilities, axis=1, keepdims=True)


def main_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


assert SELECTION_METRIC == "macro_f1"
assert SELECTION_METRIC in compute_metrics((np.zeros((1, len(id2label))), np.array([0])))

## Модели BERT + LoRA

Каждая конфигурация sweep получает заново инициализированную базовую модель при одном и том же seed. Для BERT-классификатора обучаемым и сохраняемым дополнительным модулем является classifier.

Для encoder-only BERT ожидаются `attention.self.query` и `attention.self.value` в каждом encoder layer; именно их suffix-имена затем получает PEFT.


In [ ]:
NUM_LABELS = len(id2label)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def new_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_BERT,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    )


set_global_seed(SEED)
probe_base_model = new_base_model()
head_modules_to_save = ["classifier"]
assert hasattr(probe_base_model, "classifier"), "Expected BERT classifier head was not found."

LORA_TARGET_MODULES = ["query", "value"]
discovered_lora_targets = [
    module_name
    for module_name, _ in probe_base_model.named_modules()
    if module_name.endswith((".attention.self.query", ".attention.self.value"))
]
expected_target_count = 2 * probe_base_model.config.num_hidden_layers
assert len(discovered_lora_targets) == expected_target_count, (
    "Unexpected BERT query/value target modules: "
    f"found {len(discovered_lora_targets)}, expected {expected_target_count}."
)
assert all(
    module_name.endswith(tuple(LORA_TARGET_MODULES))
    for module_name in discovered_lora_targets
)
del probe_base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Device:", device)
print("Base model:", BASE_MODEL_BERT)
print("Verified BERT LoRA target count:", len(discovered_lora_targets))
print("Example targets:", discovered_lora_targets[:2])
print("BERT classification head modules to train and save:", head_modules_to_save)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
Base model: bert-base-uncased
Verified BERT LoRA target count: 24
Example targets: ['bert.encoder.layer.0.attention.self.query', 'bert.encoder.layer.0.attention.self.value']
BERT classification head modules to train and save: ['classifier']


## Fixed grid для LoRA

Для PEFT наиболее содержательны скорость обучения и ёмкость адаптера. Поэтому проверяются `learning_rate` из {1e-4, 2e-4, 3e-4 5e-4} и `r` из {8, 16}; `lora_alpha=2*r` сохраняет одинаковое масштабирование обновления. `lora_dropout=0.1`, 4 эпохи, `weight_decay=0.01`, `warmup_ratio=0.06` и batch sizes фиксированы.


In [ ]:
LORA_GRID = [
    {
        "learning_rate": learning_rate,
        "r": rank,
        "lora_alpha": 2 * rank,
        "lora_dropout": 0.1,
        "num_train_epochs": 4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
    }
    for learning_rate, rank in product([1e-4, 2e-4, 3e-4, 5e-4], [8, 16])
]

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LOGGING_STEPS = 50


def build_lora_model(config):
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config["r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=list(head_modules_to_save),
    )
    return get_peft_model(new_base_model(), lora_config)


def summarize_parameters(model, print_summary=False):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_names = [
        name for name, parameter in model.named_parameters() if parameter.requires_grad
    ]
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    trainable_share = trainable_params / total_params

    if print_summary:
        print(f"Total parameters:     {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Trainable share:      {100 * trainable_share:.4f}%")

    assert any("lora_" in name for name in trainable_names), "No trainable LoRA parameters found."
    for head_module in head_modules_to_save:
        assert any(f".{head_module}." in name for name in trainable_names), (
            f"The classification head module {head_module!r} is not trainable."
        )
    unexpected_trainable_names = [
        name
        for name in trainable_names
        if "lora_" not in name
        and not any(f".{head_module}." in name for head_module in head_modules_to_save)
    ]
    assert not unexpected_trainable_names, (
        "Unexpected trainable backbone parameters: "
        f"{unexpected_trainable_names[:10]}"
    )
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_share": trainable_share,
    }


print("Number of LoRA configurations:", len(LORA_GRID))
pd.DataFrame(LORA_GRID)


Number of LoRA configurations: 8


,learning_rate,r,lora_alpha,lora_dropout,num_train_epochs,weight_decay,warmup_ratio
0,0.0001,8,16,0.1,4,0.01,0.06
1,0.0001,16,32,0.1,4,0.01,0.06
2,0.0002,8,16,0.1,4,0.01,0.06
3,0.0002,16,32,0.1,4,0.01,0.06
4,0.0003,8,16,0.1,4,0.01,0.06
5,0.0003,16,32,0.1,4,0.01,0.06
6,0.0005,8,16,0.1,4,0.01,0.06
7,0.0005,16,32,0.1,4,0.01,0.06


### Проверка forward pass до sweep


In [ ]:
set_global_seed(SEED)
smoke_model = build_lora_model(LORA_GRID[0]).to(device)
summarize_parameters(smoke_model, print_summary=True)
smoke_model.eval()

smoke_batch_size = min(2, len(tokenized_datasets["validation"]))
smoke_batch = tokenized_datasets["validation"][:smoke_batch_size]
smoke_inputs = {
    key: smoke_batch[key].to(device)
    for key in ("input_ids", "attention_mask")
}
with torch.no_grad():
    smoke_output = smoke_model(**smoke_inputs)

expected_logits_shape = (smoke_batch_size, NUM_LABELS)
assert tuple(smoke_output.logits.shape) == expected_logits_shape
print("Forward pass logits shape:", tuple(smoke_output.logits.shape))

del smoke_model, smoke_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_re

Total parameters:     109,781,766
Trainable parameters: 297,219
Trainable share:      0.2707%
Forward pass logits shape: (2, 3)


## Hyperparameter search только по validation macro-F1


In [ ]:
assert SELECTION_METRIC == "macro_f1"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)


def make_training_arguments(config_id, config):
    training_args_kwargs = {
        "output_dir": str(CHECKPOINTS_DIR / f"config_{config_id:02d}"),
        "learning_rate": config["learning_rate"],
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "num_train_epochs": config["num_train_epochs"],
        "weight_decay": config["weight_decay"],
        "warmup_ratio": config["warmup_ratio"],
        "logging_steps": LOGGING_STEPS,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": SELECTION_METRIC,
        "greater_is_better": True,
        "save_total_limit": 1,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "fp16": torch.cuda.is_available(),
    }
    signature = inspect.signature(TrainingArguments.__init__)
    strategy_key = "eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"
    training_args_kwargs[strategy_key] = "epoch"
    return TrainingArguments(**training_args_kwargs)


sweep_records = []
best_run = None
for config_id, config in enumerate(LORA_GRID, start=1):
    print(f"\nTraining LoRA configuration {config_id}/{len(LORA_GRID)}:", config)
    set_global_seed(SEED)
    run_model = build_lora_model(config).to(device)
    parameter_summary = summarize_parameters(run_model)
    run_trainer = Trainer(
        model=run_model,
        args=make_training_arguments(config_id, config),
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    start_time = time.perf_counter()
    run_trainer.train()
    run_training_time_sec = time.perf_counter() - start_time
    validation_metrics = run_trainer.evaluate(
        tokenized_datasets["validation"],
        metric_key_prefix="validation",
    )
    assert run_trainer.state.best_model_checkpoint is not None

    record = {
        "config_id": config_id,
        **config,
        "val_macro_f1": float(validation_metrics["validation_macro_f1"]),
        "val_weighted_f1": float(validation_metrics["validation_weighted_f1"]),
        "val_accuracy": float(validation_metrics["validation_accuracy"]),
        "training_time_sec": run_training_time_sec,
        "best_checkpoint": run_trainer.state.best_model_checkpoint,
        **parameter_summary,
    }
    sweep_records.append(record)
    if best_run is None or record["val_macro_f1"] > best_run["record"]["val_macro_f1"]:
        best_run = {
            "record": dict(record),
            "config": dict(config),
            "parameter_summary": dict(parameter_summary),
        }

    del run_trainer, run_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sweep_results_df = (
    pd.DataFrame(sweep_records)
    .sort_values("val_macro_f1", ascending=False)
    .reset_index(drop=True)
)
sweep_results_df.to_csv(SWEEP_RESULTS_PATH, index=False)

best_config = best_run["config"]
selected_training_time_sec = best_run["record"]["training_time_sec"]
print("\nBest LoRA configuration selected only on validation macro-F1:")
print(best_config)
print("Best checkpoint:", best_run["record"]["best_checkpoint"])
sweep_results_df



Training LoRA configuration 1/8: {'learning_rate': 0.0001, 'r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1, 'num_train_epochs': 4, 'weight_decay': 0.01, 'warmup_ratio': 0.06}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.637853,0.635724,0.740042,0.484510,0.681933
2,0.618055,0.633944,0.733054,0.467760,0.669352
3,0.616077,0.585522,0.761705,0.579267,0.731211
4,0.580173,0.579471,0.758211,0.584707,0.730218


[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.477066,0.452324,0.829490,0.768661,0.825572
2,0.404832,0.409899,0.845563,0.784519,0.837895
3,0.325512,0.385188,0.857442,0.817496,0.857894
4,0.250327,0.395590,0.865828,0.826836,0.865213


[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.250327,0.395590,4,0.865828,0.826836,0.865213



Best LoRA configuration selected only on validation macro-F1:
{'learning_rate': 0.0005, 'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.1, 'num_train_epochs': 4, 'weight_decay': 0.01, 'warmup_ratio': 0.06}
Best checkpoint: /content/LORA-course-project/artifacts/bert_lora_hparam_search/checkpoints/config_08/checkpoint-2028


,config_id,learning_rate,r,lora_alpha,lora_dropout,num_train_epochs,weight_decay,warmup_ratio,val_macro_f1,val_weighted_f1,val_accuracy,training_time_sec,best_checkpoint,total_params,trainable_params,trainable_share
0,8,0.0005,16,32,0.1,4,0.01,0.06,0.826836,0.865213,0.865828,178.162146,/content/LORA-course-project/artifacts/bert_lo...,110076678,592131,0.005379
1,6,0.0003,16,32,0.1,4,0.01,0.06,0.814563,0.857596,0.859539,177.969411,/content/LORA-course-project/artifacts/bert_lo...,110076678,592131,0.005379
2,7,0.0005,8,16,0.1,4,0.01,0.06,0.809480,0.851975,0.853948,177.422985,/content/LORA-course-project/artifacts/bert_lo...,109781766,297219,0.002707
3,5,0.0003,8,16,0.1,4,0.01,0.06,0.792424,0.840994,0.843466,177.099596,/content/LORA-course-project/artifacts/bert_lo...,109781766,297219,0.002707
4,4,0.0002,16,32,0.1,4,0.01,0.06,0.791889,0.841669,0.844165,177.570566,/content/LORA-course-project/artifacts/bert_lo...,110076678,592131,0.005379
5,3,0.0002,8,16,0.1,4,0.01,0.06,0.771888,0.829341,0.832984,185.098686,/content/LORA-course-project/artifacts/bert_lo...,109781766,297219,0.002707
6,2,0.0001,16,32,0.1,4,0.01,0.06,0.619587,0.752337,0.772187,184.877009,/content/LORA-course-project/artifacts/bert_lo...,110076678,592131,0.005379
7,1,0.0001,8,16,0.1,4,0.01,0.06,0.584707,0.730218,0.758211,203.643760,/content/LORA-course-project/artifacts/bert_lo...,109781766,297219,0.002707


## Единственная финальная оценка на test split

Загружаем checkpoint победившей validation-конфигурации заново


In [ ]:
assert FINAL_TEST_EVALUATION_ONLY is True
assert best_run is not None

set_global_seed(SEED)
best_base_model = new_base_model()
best_model = PeftModel.from_pretrained(
    best_base_model,
    best_run["record"]["best_checkpoint"],
    is_trainable=False,
).to(device)

final_eval_args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR / "final_evaluation"),
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    report_to="none",
    fp16=torch.cuda.is_available(),
)
best_trainer = Trainer(model=best_model, args=final_eval_args, compute_metrics=compute_metrics)
test_prediction_output = best_trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)
test_logits = test_prediction_output.predictions
test_probabilities = softmax_numpy(test_logits)
test_predictions = np.argmax(test_probabilities, axis=1)
test_labels = raw_dataframes["test"]["label"].to_numpy()
test_metrics = main_metrics(test_labels, test_predictions)

class_names = [id2label[class_id] for class_id in sorted(id2label)]
test_classification_report = classification_report(
    test_labels,
    test_predictions,
    labels=sorted(id2label),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
test_classification_report_df = (
    pd.DataFrame(test_classification_report).transpose().rename_axis("class").reset_index()
)
test_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(test_labels, test_predictions, labels=sorted(id2label)),
    index=[f"true_{class_name}" for class_name in class_names],
    columns=[f"pred_{class_name}" for class_name in class_names],
)

test_predictions_df = raw_dataframes["test"].copy()
test_predictions_df["true_label"] = test_labels
test_predictions_df["true_label_name"] = test_predictions_df["true_label"].map(id2label)
test_predictions_df["pred_label"] = test_predictions
test_predictions_df["pred_label_name"] = test_predictions_df["pred_label"].map(id2label)
test_predictions_df["correct"] = test_predictions_df["true_label"] == test_predictions_df["pred_label"]
for class_id, class_name in id2label.items():
    test_predictions_df[f"prob_{class_name.lower()}"] = test_probabilities[:, class_id]

print("Final test metrics from the validation-selected LoRA checkpoint:")
test_metrics


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Final test metrics from the validation-selected LoRA checkpoint:


{'accuracy': 0.8701842546063652,
 'macro_f1': 0.8284152512235106,
 'weighted_f1': 0.8704869805790268}

## Сохранение результатов и лучшего adapter checkpoint

training_time_sec в ней относится только к обучению выбранной validation-конфигурации, а не к длительности всего sweep.


In [ ]:
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)
BEST_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

FINAL_RESULT_COLUMNS = [
    "model",
    "adaptation",
    "test_macro_f1",
    "test_weighted_f1",
    "test_accuracy",
    "bearish_f1",
    "bullish_f1",
    "neutral_f1",
    "total_params",
    "trainable_params",
    "trainable_share",
    "training_time_sec",
]

selected_parameter_summary = best_run["parameter_summary"]
final_test_row = {
    "model": BASE_MODEL_BERT,
    "adaptation": "LoRA",
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "bearish_f1": float(test_classification_report["Bearish"]["f1-score"]),
    "bullish_f1": float(test_classification_report["Bullish"]["f1-score"]),
    "neutral_f1": float(test_classification_report["Neutral"]["f1-score"]),
    "total_params": selected_parameter_summary["total_params"],
    "trainable_params": selected_parameter_summary["trainable_params"],
    "trainable_share": selected_parameter_summary["trainable_share"],
    "training_time_sec": selected_training_time_sec,
}
final_test_results_df = pd.DataFrame([final_test_row], columns=FINAL_RESULT_COLUMNS)
assert list(final_test_results_df.columns) == FINAL_RESULT_COLUMNS
final_test_results_df.to_csv(FINAL_TEST_RESULTS_PATH, index=False)

test_predictions_df.to_csv(
    RUN_TABLES_DIR / "bert_lora_best_hparam_test_predictions.csv", index=False
)
test_classification_report_df.to_csv(
    RUN_TABLES_DIR / "bert_lora_best_hparam_test_classification_report.csv", index=False
)
test_confusion_matrix_df.to_csv(
    RUN_TABLES_DIR / "bert_lora_best_hparam_test_confusion_matrix.csv"
)

best_model.save_pretrained(BEST_ADAPTER_DIR)
tokenizer.save_pretrained(BEST_ADAPTER_DIR)
experiment_config = {
    "dataset_name": DATASET_NAME,
    "model": BASE_MODEL_BERT,
    "adaptation": "LoRA",
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "selection_metric": SELECTION_METRIC,
    "selection_split": "validation",
    "test_evaluation_after_validation_selection_only": FINAL_TEST_EVALUATION_ONLY,
    "fixed_grid": LORA_GRID,
    "fixed_training_settings": {
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "target_modules": LORA_TARGET_MODULES,
        "modules_to_save": head_modules_to_save,
    },
    "selected_config": best_config,
    "selected_best_checkpoint": best_run["record"]["best_checkpoint"],
    "selected_training_time_sec": selected_training_time_sec,
    "sweep_results_csv": str(SWEEP_RESULTS_PATH),
    "final_test_results_csv": str(FINAL_TEST_RESULTS_PATH),
}
with EXPERIMENT_CONFIG_PATH.open("w", encoding="utf-8") as config_file:
    json.dump(experiment_config, config_file, indent=2)

print("Saved validation sweep table to:", SWEEP_RESULTS_PATH)
print("Saved final best-model test table to:", FINAL_TEST_RESULTS_PATH)
print("Saved best LoRA adapter and tokenizer to:", BEST_ADAPTER_DIR)
final_test_results_df


Saved validation sweep table to: /content/LORA-course-project/reports/tables/bert_lora/bert_lora_hparam_search_validation_results.csv
Saved final best-model test table to: /content/LORA-course-project/reports/tables/bert_lora/bert_lora_best_hparam_test_results.csv
Saved best LoRA adapter and tokenizer to: /content/LORA-course-project/artifacts/bert_lora_hparam_search/best_adapter


,model,adaptation,test_macro_f1,test_weighted_f1,test_accuracy,bearish_f1,bullish_f1,neutral_f1,total_params,trainable_params,trainable_share,training_time_sec
0,bert-base-uncased,LoRA,0.828415,0.870487,0.870184,0.766854,0.805112,0.91328,110076678,592131,0.005379,178.162146
